[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/openai_sessions_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">配置会话</h1>

会话由一系列追踪组成，表示一次会话或一个对话线程。每个回答都有独立的追踪，这些追踪通过共同所属的会话关联起来。
要将追踪关联，需要传入一个特殊的元数据键，其值是该线程的唯一标识符。

本教程使用 OpenAI 和 OpenInference 埋点配置会话。

## 1. 安装依赖并导入库

安装依赖。

In [ ]:
!pip install "openai>=1.0.0" arize-phoenix jsonschema openinference-instrumentation-openai openinference-instrumentation opentelemetry-api opentelemetry-sdk openinference-semantic-conventions

## 配置 OpenAI API 密钥并实例化 OpenAI 客户端

如果尚未通过环境变量设置 OpenAI API 密钥，请在此设置。

In [ ]:
import os
from getpass import getpass

from openai import OpenAI

if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥： ")
client = OpenAI(api_key=openai_api_key)

## 为 OpenAI 客户端添加埋点

使用以 OpenInference 格式发送遥测数据的追踪器，为 OpenAI 客户端添加埋点。[OpenInference](https://arize-ai.github.io/open-inference-spec/trace/) 是用于捕获和存储 LLM 应用追踪的开放标准，可使应用无缝接入 Phoenix 等 LLM 可观测性解决方案。

In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from phoenix.otel import register

tracer_provider = register(project_name="openai-sessions-example")
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

## 在后台运行 Phoenix

以后台会话启动 Phoenix，收集添加埋点的 OpenAI 客户端发出的追踪数据。生产环境应在容器中运行 Phoenix。

In [ ]:
import phoenix as px

px.launch_app().view()

## 创建最简智能体

创建使用 OpenAI 的智能体框架。

In [ ]:
import uuid

from openinference.instrumentation import using_session
from openinference.semconv.trace import SpanAttributes
from opentelemetry import trace

session_id = str(uuid.uuid4())

tracer = trace.get_tracer(__name__)


@tracer.start_as_current_span(
    name="agent", attributes={SpanAttributes.OPENINFERENCE_SPAN_KIND: "agent"}
)
def assistant(
    messages: list[dict],
    session_id: str,
):
    current_span = trace.get_current_span()
    current_span.set_attribute(SpanAttributes.SESSION_ID, session_id)
    current_span.set_attribute(SpanAttributes.INPUT_VALUE, messages[-1].get("content"))

    # 将 session_id 传递给 OpenAI 插桩创建的子跨度
    # 此步骤不是必需的，但有助于将跨度关联到同一会话
    with using_session(session_id):
        response = (
            client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "system", "content": "你是乐于助人的助手。"}] + messages,
            )
            .choices[0]
            .message
        )

    current_span.set_attribute(SpanAttributes.OUTPUT_VALUE, response.content)
    return response


messages = [{"role": "user", "content": "hi! im bob"}]
response = assistant(
    messages,
    session_id=session_id,
)
messages = messages + [response, {"role": "user", "content": "what's my name?"}]
response = assistant(
    messages,
    session_id=session_id,
)
messages = messages + [response, {"role": "user", "content": "what's 4+5?"}]
response = assistant(
    messages,
    session_id=session_id,
)